# 第22课 · 教计算机"感受变化"——导数（derivative）、切线斜率与数值微分的中心差分

**目标**：导数就是「输入动一点，输出动多少」——曲线的斜率。

**为什么对 Aurora 重要**：`numeric_derivative` 是验证反向传播（backpropagation，BP）正确性的标准工具——用数值梯度（numerical gradient）和解析梯度（analytical gradient）逐元素比对，确认 MFCC（Mel-Frequency Cepstral Coefficients，MFCC）、梅尔滤波器（mel filterbank）等模块的求导实现没有错误。

← **上一课**　[L21 · 矩阵即滤波](../2_linear_algebra/L21_aurora_matrices.ipynb)

> 上节课学习了 **矩阵即滤波**：DFT 矩阵 / Mel 矩阵：音频处理 = 矩阵乘法。  
> 本课将探讨 **导数**。

## 本课剧情：怎么教计算机"感受变化"？

人类直觉上知道"坡度"——站在山坡上脚感受到倾斜。  
但计算机只有数字。它怎么知道一个函数在某点的"倾斜程度"？

方法：找两个离 x 很近的点，做一条割线（secant line），用割线斜率近似切线斜率（tangent slope）：

```
f'(x) ≈ (f(x+h) - f(x-h)) / (2h)     ← 中心差分公式
```

两点越近（h 越小），割线越接近切线。但 h 不能无限小——计算机浮点数（floating-point number）有精度下限（约 1e-15），h 太小会产生"灾难性消去"（catastrophic cancellation），误差反而变大。

最优 h ≈ 1e-5：不太大不太小，平衡截断误差（truncation error）和舍入误差（rounding error）。

为什么这对 Aurora 重要？  
梯度下降（L25）需要在每个参数处计算导数。数值导数 `numeric_derivative` 是"万能的"梯度检验器——验证你的解析公式有没有写错，是 ML 系统调试的基本工具。

## 学习目标

1. **陈述极限定义**：写出 f'(x) = lim_{h→0} (f(x+h)-f(x))/h，解释为何计算机必须用有限 h 近似。
2. **推导 O(h²) 误差**：说明中心差分（central difference）为何比前向差分（forward difference）高一阶精度——误差从 O(h) 降到 O(h²)，h=1e-5 时误差小约 5 个数量级（泰勒展开（Taylor expansion）消去一阶项）。
3. **实现 `numeric_derivative`**：完成 TODO stub，用中心差分在任意点求导数近似值。
4. **实验找最优 h**：通过 h-sweep 实验观察误差先降后升的 U 形曲线，确认最优步长约 1e-5～1e-7。

## 1. 中心差分：用两点近似斜率

**为什么用中心差分而不是前向差分？**

前向差分：`f'(x) ≈ (f(x+h) - f(x)) / h`，误差阶 O(h)  
中心差分：`f'(x) ≈ (f(x+h) - f(x-h)) / (2h)`，误差阶 O(h²)

h=1e-5 时，前向误差 ≈ 1e-5，中心差分误差 ≈ 1e-10——小了 **5 个数量级**。

手算验证（f(x)=x², x=1, h=0.1）：
```
CD = (f(1.1) - f(0.9)) / (0.2)
   = (1.21 - 0.81) / 0.2
   = 0.40 / 0.2
   = 2.0      ← 与真值 f'(1) = 2×1 = 2 完全相同！
```

**为什么 h 不能无限小？**
浮点数有机器精度 ε ≈ 2.2×10⁻¹⁶。当 h < √ε ≈ 1e-8 时，分子 `f(x+h)-f(x-h)` 与 `f(x)` 相比极小，有效位数丧失（catastrophic cancellation）。

### 极限定义（limit definition）

导数的严格定义来自极限：

$$f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$$

意思是：h 越接近 0，割线（secant line）的斜率就越接近切线（tangent line）的斜率。

**但计算机无法取 h→0**：浮点数精度有限（约 1e-16），h 太小时分子 `f(x+h)-f(x-h)` 两个近似相等的数相减，有效位数急剧损失（灾难性消除，catastrophic cancellation）。

因此实践中用有限 h（通常 h≈1e-5）的**中心差分**来近似：

$$f'(x) \approx \frac{f(x+h) - f(x-h)}{2h}, \quad h \approx 10^{-5}$$

这将误差从 O(h)（前向差分）压缩到 O(h²)（中心差分），h=1e-5 时误差约 1e-10。

## 实验入口：用数值变化观察函数

下面的实验在 x=3.0 处计算 f(x)=x² 的数值导数，关注 `h` 缩小时计算值如何接近真实斜率（真值=2x=6）。

In [ ]:
import numpy as np
f = lambda x: x**2
h = 1e-5
x = 3.0
print('f(x)=x^2 在 x=3 的斜率 ≈', (f(x+h)-f(x-h))/(2*h), ' (真值 2x=6)')

In [ ]:
# 切线可视化：函数曲线 + 切线
import numpy as np
import matplotlib.pyplot as plt

x_arr = np.linspace(0, 5, 300)
f_arr = x_arr**2

x0 = 3.0
slope = 2 * x0  # f'(x0) = 2x0 = 6（解析值）
tangent = f(x0) + slope * (x_arr - x0)  # 切线方程

plt.figure(figsize=(6, 4))
plt.plot(x_arr, f_arr, label=r"$f(x)=x^2$", linewidth=2)
plt.plot(x_arr, tangent, "--", label=f"切线 @ x={x0}（斜率={slope}）", linewidth=1.5)
plt.scatter([x0], [x0**2], color="red", zorder=5, label=f"x={x0}")
plt.xlim(0, 5); plt.ylim(-5, 25)
plt.xlabel("x"); plt.ylabel("f(x)")
plt.title("切线斜率 = 导数值")
plt.legend(); plt.tight_layout(); plt.show()
print(f"x={x0} 处：解析斜率={slope}，数值斜率（若已实现）将在验证 cell 中对比")

## 动手观察：多点斜率一览

对 f(x)=x² 在 xs=[-2,-1,0,1,2] 五个点上批量计算数值斜率（向量化中心差分），与真值 2x 逐点比较，验证误差一致在 1e-3 量级（此处 h=1e-3）。

In [ ]:
import numpy as np

def f(x):
    return x**2

xs = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
h = 1e-3
slopes = (f(xs + h) - f(xs - h)) / (2 * h)

print('x =', xs)
print('f(x) =', f(xs))
print('近似斜率 =', np.round(slopes, 3))
print('理论斜率 2x =', 2 * xs)


## 代码实验：遍历不同位置，看斜率如何变化

对 f(x)=x²+2x 在 [-3, 3] 上取 7 个点，逐点打印数值导数与解析导数（2x+2）的差值，验证中心差分在整个区间都准确。

In [ ]:
import numpy as np

def f(x):
    return x**2 + 2*x

h = 1e-4
for x in np.linspace(-3, 3, 7):
    slope = (f(x + h) - f(x - h)) / (2*h)
    print(f'x={x:5.2f} | f(x)={f(x):6.2f} | slope≈{slope:6.2f}')


## 2. ✏️ 实现 `numeric_derivative(f, x, h=1e-5)`

**推理路线**：
1. 函数签名：接受 `f`（任意可调用）、`x`（float，求导点）、`h`（扰动步长，默认 1e-5）。
2. 计算 `f(x+h)` 和 `f(x-h)`——各向右/左扰动 h，共两次函数求值。
3. 两者相减后除以 `2*h`，得到中心差分近似，返回单个 float。

**参考输入输出**：f=sin, x=0, h=1e-5 → `(sin(1e-5) - sin(-1e-5))/(2e-5)` ≈ `(1e-5 - (-1e-5))/(2e-5)` = 1.0 = cos(0) ✓

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


### 写代码前，先把变量表补完整

写 `numeric_derivative` 前明确三件事：
- 输入：`f`（可调用函数）、`x`（求导点）、`h`（扰动步长，默认 1e-5）
- 关键步骤：计算 `f(x+h)` 和 `f(x-h)`，相减后除以 `2h`
- 返回：单个浮点数，表示 f 在 x 处的斜率近似值

In [ ]:
def numeric_derivative(f, x, h=1e-5):
    # ✏️ TODO: 中心差分近似导数
    raise NotImplementedError("TODO: 实现中心差分近似 — 用 (f(x+h) - f(x-h)) / (2*h)")

In [ ]:
try:
    assert abs(numeric_derivative(lambda x: x**2, 3.0) - 6.0) < 1e-8, "x²在x=3处导数应≈6.0，误差应<1e-8"
    assert abs(numeric_derivative(np.sin, 0.0) - 1.0) < 1e-8, "sin在x=0处导数应≈1.0，误差应<1e-8"
    print("✅ 通过：你会数值求导了。")
except (NotImplementedError, TypeError):
    print("⚠️ 还没实现 numeric_derivative，请完成 TODO 后重新运行。")
except AssertionError as e:
    print(f"❌ 断言失败：{e}")

## 3. 记几个常见导数（背下来省事）

| f(x) | f'(x) |
|---|---|
| xⁿ | n·xⁿ⁻¹ |
| eˣ | eˣ |
| sin x | cos x |
| ln x | 1/x |

**🔗 Aurora 连接**：`numeric_derivative` 将在后续里程碑中集成到 `src/aurora/audio/grad_check.py`（尚未创建），用于将 MFCC、梅尔滤波器等模块的解析梯度与数值梯度逐元素比对，差异超过 1e-5 即判定求导实现有误。目前可在当前 notebook 中直接使用实现的函数进行验证。下一节（`L23_gradients.ipynb`）把单点斜率推广到多变量：沿每个参数维度重复一次 `numeric_derivative`，拼成梯度向量。

In [ ]:
# 参数实验：h 的最优区间（与 cell[17] 描述对应）
hs = [1e-1, 1e-2, 1e-3, 1e-5, 1e-7, 1e-9, 1e-11, 1e-13, 1e-15]
try:
    errs = [abs(numeric_derivative(np.sin, 1.0, h) - np.cos(1.0)) for h in hs]  # 精确值 cos(1)
except (NotImplementedError, TypeError):
    print("⬜ 请先完成上面的 numeric_derivative 练习，再运行本格")
else:
    print(f"{'h':>10}  {'|误差|':>12}")
    print('-' * 26)
    for h, err in zip(hs, errs):
        print(f"{h:10.0e}  {err:12.2e}")
    print("\n最优 h 约 1e-5～1e-7，误差约 1e-11；h 继续缩小后浮点舍入误差反而上升。")


## 参数实验：h 的最优区间

把 `h` 从 1e-1 逐步减小到 1e-15，每步打印 `abs(numeric_derivative(np.sin, 1.0, h) - np.cos(1.0))`，观察误差先减后增的 U 形曲线，找到最优步长约 1e-5~1e-7。

- **h 偏大（1e-1 到 1e-3）**：截断误差主导，误差约 h²/6·f'''(x)，随 h 缩小以平方速率下降。
- **h 最优（1e-5 到 1e-7）**：截断误差与浮点舍入误差平衡，总误差最小约 1e-11。
- **h 过小（1e-12 以下）**：`f(x+h) - f(x-h)` 两个近似相等的数相减，有效位数从 16 位骤降至 4 位以下，误差反而随 h 减小而上升。

## 本课收束

现在可以用 `numeric_derivative(f, x, h=1e-5)` 在任意点计算可调用函数的斜率，输出单个浮点数，误差量级 1e-10。这个函数在 Aurora 梯度检验（gradient check）中直接使用：将损失对权重的解析梯度与数值梯度逐元素比对，差异超过 1e-5 即报错。`h` 的最优范围在 1e-5 附近，过大则截断误差主导，过小则浮点舍入误差主导。

下一课：**L23**（梯度）把这个操作沿每个参数维度重复，得到多变量函数的梯度向量。

---
⬇️ **通关检验**：收束小结已读；请完成下方白板挑战后再勾选自评。


## ✏️ 白板挑战：数值导数手算（目标 8 分钟）

盖上屏幕，纸上作答：

**问 1**：f(x) = x³，解析导数 f'(x) = ?，在 x=2 处 f'(2) = ?

**问 2**：用中心差分公式（h=0.1），手算 f(x)=x² 在 x=1 处的数值导数：  
CD = (f(1.1) - f(0.9)) / 0.2 = ?

**问 3**：f(x) = sin(x)，f'(0) = ?（根据常见导数表）

**问 4**：`h` 从 1e-5 减小到 1e-15 时，数值误差会变大还是变小？为什么？

推导完成后运行下面格对答案。

In [ ]:
# ✏️ 对答案格
import numpy as np

# 问1：f'(x) = 3x², f'(2) = 12
f1 = lambda x: x**3
df1_analytical = lambda x: 3 * x**2
assert np.isclose(df1_analytical(2), 12.0)
try:
    nd = numeric_derivative(f1, 2.0)
    assert abs(nd - 12.0) < 1e-7, f"x³ 在 x=2 数值导数应≈12，得到 {nd}"
    print(f"Q1 ✅  f(x)=x³，f'(x)=3x²，f'(2)={df1_analytical(2)}，数值验证={nd:.8f}")
except (NotImplementedError, TypeError):
    print("⬜ Q1：请先实现 numeric_derivative()，再运行对答案格")

# 问2：CD 手算 — f(x)=x², x=1, h=0.1
f2 = lambda x: x**2
h = 0.1
cd = (f2(1.1) - f2(0.9)) / (2*h)
assert np.isclose(cd, 2.0, atol=1e-10), f"CD 应为2，得到{cd}"
print(f"Q2 ✅  CD = (1.21-0.81)/0.2 = {cd:.4f}  （真值 f'(1)=2）")

# 问3：sin'(0) = cos(0) = 1
assert np.isclose(np.cos(0), 1.0)
print(f"Q3 ✅  f(x)=sin(x)，f'(0)=cos(0)={np.cos(0):.4f}")

# 问4：h 太小时误差增大（灾难性消去，需在 f(x±h) 相近的非零点观察，取 x=1）
errors = []
for h_val in [1e-5, 1e-8, 1e-12, 1e-15]:
    nd_sin = (np.sin(1.0 + h_val) - np.sin(1.0 - h_val)) / (2*h_val)
    errors.append(abs(nd_sin - np.cos(1.0)))
# 最优h(1e-5)误差最小，1e-15误差最大
assert errors[0] < errors[-1], "h越小误差越大的现象未观察到"
print(f"Q4 ✅  h=1e-5误差={errors[0]:.2e}，h=1e-15误差={errors[-1]:.2e}")
print("     h太小→浮点消去误差↑；最优h≈1e-5 平衡截断误差和舍入误差")
print("\n🎉 导数白板挑战通过！数值微分=计算机感受变化的方式 已内化。")

In [ ]:
# ✏️ 本课自评
l22_review = {
    "central_diff_formula":       None,  # 记住 CD=(f(x+h)-f(x-h))/(2h)？True/False
    "numeric_derivative_done":    None,  # numeric_derivative 实现并通过断言？True/False
    "optimal_h_range":            None,  # 知道最优 h≈1e-5 及原因？True/False
    "common_derivatives":         None,  # 记住 xⁿ/eˣ/sin/cos 的导数？True/False
    "whiteboard_passed":          None,  # 白板挑战纸上推导完成？True/False
}

unfilled = [k for k, v in l22_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l22_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L22 全部通关！进入 L23：梯度')

---

→ **下一课**　[L23 · 梯度](L23_gradients.ipynb)

> 下节课将学习 **梯度**：多元函数的"最陡上坡"方向，偏导与梯度向量的计算。